# 01 — Raw cytometry files to Stage 1 HDF5

This notebook audits CSV manifests, builds one patient-centric Stage 1 file for multiple datasets/cell-count configurations, and verifies every stored matrix. Set `RUN_BUILD=True` only after the manifest audit has no errors.

In [ ]:
from pathlib import Path
import h5py
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from flowlot.io import audit_manifest, audit_stage1, build_stage1_from_manifest

## Configuration

Each manifest requires `patient_id,tube_id,path,label`; `markers` is optional and uses semicolon-separated names. Paths may be absolute or relative to the manifest. Add one entry below for every dataset and requested subsampling level.

In [ ]:
CONFIGURATIONS = [
    {'dataset': 'BLAST110', 'manifest': Path('../manifests/blast110.csv'), 'cells': 1000},
    # {'dataset': 'FlowCAPII', 'manifest': Path('../manifests/flowcapii.csv'), 'cells': 1000},
]
STAGE1 = Path('../data/stage1_raw_data.h5')
SEED = 42
RUN_BUILD = False  # Safety switch: audit first.

In [ ]:
template = pd.DataFrame([
    {'patient_id': 'P001', 'tube_id': 'P1', 'path': '../raw/P001_P1.fcs', 'label': 'NBM', 'markers': ''},
    {'patient_id': 'P001', 'tube_id': 'P2', 'path': '../raw/P001_P2.npy', 'label': 'NBM', 'markers': 'FSC-A;SSC-A;CD45'},
])
display(template)  # template.to_csv('../manifests/template.csv', index=False)

## Preflight manifest audit

This detects missing/unsupported inputs, empty fields, duplicate patient–tube keys, and inconsistent labels before HDF5 writing begins.

In [ ]:
manifest_inventories, manifest_issues = [], []
for config in CONFIGURATIONS:
    inventory, issues = audit_manifest(config['manifest'])
    inventory = inventory.assign(dataset=config['dataset'], requested_cells=str(config['cells']))
    issues = issues.assign(dataset=config['dataset'], requested_cells=str(config['cells']))
    manifest_inventories.append(inventory)
    manifest_issues.append(issues)
manifest_inventory = pd.concat(manifest_inventories, ignore_index=True)
manifest_issue_table = pd.concat(manifest_issues, ignore_index=True)
display(manifest_issue_table)
if {'dataset', 'requested_cells', 'label', 'tube_id'}.issubset(manifest_inventory.columns):
    display(manifest_inventory.groupby(['dataset', 'requested_cells', 'label', 'tube_id']).size().rename('files').reset_index())
if not manifest_issue_table.empty and (manifest_issue_table['severity'] == 'error').any():
    raise ValueError('Correct manifest errors before setting RUN_BUILD=True')

## Build Stage 1

The first configuration creates the file; later configurations append independent dataset/cell-count groups. Subsampling is seeded and without replacement.

In [ ]:
if RUN_BUILD:
    for index, config in enumerate(CONFIGURATIONS):
        build_stage1_from_manifest(
            config['manifest'], STAGE1, config['dataset'], config['cells'],
            seed=SEED, mode='w' if index == 0 else 'a',
        )
    print('Created', STAGE1.resolve())
else:
    print('Dry run. Set RUN_BUILD=True after the audit passes.')

## Post-build verification and per-dataset statistics

The audit reads matrices in chunks and checks schema, shape/marker agreement, duplicate markers, finite values, original counts, and cross-tube labels.

In [ ]:
if STAGE1.exists():
    stage1_inventory, stage1_issues = audit_stage1(STAGE1)
    summary = stage1_inventory.groupby(['dataset', 'cell_count', 'tube', 'label']).agg(
        patients=('patient_id', 'nunique'), stored_cells=('n_cells', 'sum'),
        median_cells=('n_cells', 'median'), markers=('n_markers', 'first'),
        finite_fraction=('finite_fraction', 'min'),
    ).reset_index()
    display(summary)
    display(stage1_issues)
    coverage = stage1_inventory.pivot_table(index=['dataset', 'cell_count', 'patient_id'], columns='tube', values='n_cells', aggfunc='size', fill_value=0)
    display(coverage.head())
    sns.catplot(data=stage1_inventory, x='tube', y='n_cells', col='dataset', kind='box', sharey=False)
    plt.show()
    assert stage1_issues.empty, 'Stage 1 verification failed; inspect stage1_issues'
else:
    print('Stage 1 does not exist yet.')

In [ ]:
if STAGE1.exists():
    with h5py.File(STAGE1) as handle:
        print('schema:', dict(handle.attrs))
        for dataset in handle:
            for cells in handle[dataset]:
                print(f'/{dataset}/{cells}: {len(handle[f"{dataset}/{cells}"])} patient groups')